In [1]:
from z3 import *
from building_blocks.set import *
from building_blocks.mapping import *
from building_blocks.props import *

from z3 import Solver, IntSort, BoolRef, SortRef, ArrayRef, ModelRef, Const, DeclareSort
from building_blocks.base import EvalAt, IGuarded
from building_blocks.fin import Fin
from building_blocks.set import Set, SetRef, Elements
from building_blocks.mapping import Mapping, MappingRef
from building_blocks.props import ForAll, Exists, Prod, Sig, Injective

import z3_monkey
z3_monkey.mk_mapping = MappingSort
    

z3.main_ctx().guarded = IGuarded.Context()

In [2]:
from dataclasses import dataclass
from functools import cmp_to_key


@dataclass
class Subdomain:
    sort: SortRef
    univ: SortRef
    proj: ArrayRef  # f : sort -> univ
    inj: BoolRef    # Injective f

    @classmethod
    def mk(cls, name, univ):
        sort = DeclareSort(name)
        f = Const(f"{name}.proj", sort ** univ)
        return cls(sort, univ, f, Injective(f))


@dataclass
class ListRef(EvalAt, IGuarded):
    dom: SetRef       # ι : Set  (index domain)
    ord: ArrayRef     # R : ι -> ι -> B
    val: ArrayRef     # val : ι -> a
    linord: BoolRef   # LinearOrder R

    def guards(self): return [self.linord]

    def eval(self, m: ModelRef):
        return [m.eval(self.val[i]) for i in ordered(self.dom, self.ord, m)]


def List(name, value_sort):
    I = Set(f"{name}.ι", U)
    R = Mapping(f"{name}.ord", I ** I ** bool)
    f = Mapping(f"{name}.val", I ** value_sort)
    return ListRef(I, R, f, LinearOrder(R))


def comparator(R: dict | MappingRef, m: ModelRef=None):
    ev = m.eval if m else lambda x: x
    def cmp(x, y):
        if x == y: return 0
        if ev(R[x][y]): return -1
        return 1
    return cmp

def ordered(l: list | set | SetRef, R: dict | MappingRef, m: ModelRef=None):
    if isinstance(l, SetRef): l = l @ m
    return sorted(l, key=cmp_to_key(comparator(R, m)))


#U = DeclareSort("○")
U = Fin[3]
A = IntSort() #DeclareSort("α")


l1 = List("l₁", A)
l1

ListRef(dom=l₁.ι, ord=l₁.ord, val=l₁.val, linord=And(And(And(ForAll(x, Implies(l₁.ι[x], l₁.ord[x][x])),
            ForAll([x, y, z],
                   Implies(l₁.ι[x],
                           Implies(l₁.ι[y],
                                   Implies(l₁.ι[z],
                                        Implies(l₁.ord[x][y],
                                        Implies(l₁.ord[y][z],
                                        l₁.ord[x][z]))))))),
        ForAll([x, y],
               Implies(l₁.ι[x],
                       Implies(l₁.ι[y],
                               Implies(l₁.ord[x][y],
                                       Implies(l₁.ord[y][x],
                                        x == y)))))),
    ForAll([x, y],
           Implies(l₁.ι[x],
                   Implies(l₁.ι[y],
                           Or(l₁.ord[x][y], l₁.ord[y][x]))))))

In [3]:

def sublist(l1, l2):
    i1, i2 = l1.dom, l2.dom
    R1, R2 = l1.ord, l2.ord
    f = Mapping('f', i1 ** i2)
    x, y = Elements('x y', i1)
    return Exists([f], f.guards() & Injective(f) &
                  ForAll([x, y], R1[x][y] == R2[f[x]][f[y]]) &
                  ForAll([x], l1.val[x] == l2.val[f[x]]))

l2 = List("l₂", A)
sublist(l1, l2)

Exists(f,
       And(ForAll(x,
                  Implies(l₁.ι[x],
                          Exists(y, And(l₂.ι[y], f[x] == y)))),
           And(And(And(ForAll(x,
                              Implies(l₁.ι[x],
                                      Exists(y,
                                        And(l₂.ι[y],
                                        f[x] == y)))),
                       ForAll([x, y],
                              Implies(l₁.ι[x],
                                      Implies(l₁.ι[y],
                                        Implies(f[x] == f[y],
                                        x == y))))),
                   ForAll([x, y],
                          Implies(l₁.ι[x],
                                  Implies(l₁.ι[y],
                                        l₁.ord[x][y] ==
                                        l₂.ord[f[x]][f[y]])))),
               ForAll(x,
                      Implies(l₁.ι[x],
                              l₁.val[x] == l₂.val[f[x]])))))

In [4]:
l3 = List("l₃", A)


s = Solver()

s.add(*l1.guards(), *l2.guards(), *l3.guards())

f1 = Mapping('f1', l1.dom ** l3.dom)  # s31)
f2 = Mapping('f2', l2.dom ** l3.dom)  # set_diff(l3.dom, s31))

ai = Element('ai', l3.dom)


s.add(*IGuarded.guards([f1, f2, ai]),
      substitute_vars(sublist(l2, l3).body(), f2),
      substitute_vars(sublist(l1, l3).body(), f1),
      Prod([l3.dom], lambda e3: one_of(
          Sig([l1.dom], lambda e1: f1[e1] == e3), (ai == e3), Sig([l2.dom], lambda e2: f2[e2] == e3))),
      Prod([l1.dom], lambda i: l3.ord[f1[i]][ai]),
      Prod([l2.dom], lambda i: l3.ord[ai][f2[i]]))

# - alt formulation with an auxiliary set
#s31 = Set("S", U)
#s.add(subset(s31, l3.dom))
#f1 = Mapping('f1', l1.dom ** s31)
#f2 = Mapping('f2', l2.dom ** set_diff(l3.dom, s31))
#Isomorphism(f1) & Prod([f1.domain()], lambda i: l1.val[i] == l3.val[f1[i]]),
#Isomorphism(f2) & Prod([f2.domain()], lambda i: l2.val[i] == l3.val[f2[i]]),
#s.add(Isomorphism(s31, l1.dom) & isomorphic(set_diff(l3.dom, s31), l2.dom))

# - this is just to get a more interesting model
s.add(card_geq(l1.dom, 1),
      card_geq(l2.dom, 1),
      Injective(l3.val))


m = s.model() if (r := s.check()) == sat else None
m

((ordered(l1.dom, l1.ord, m), l1 @ m), \
 (ordered(l2.dom, l2.ord, m), l2 @ m), \
 (ordered(l3.dom, l2.ord, m), l3 @ m), f1 @ m, ai @ m, f2 @ m) if m else r

(([⓪], [0]), ([⓪], [1]), ([⓪, ①, ②], [0, 2, 1]), {⓪: ②}, ⓪, {⓪: ①})

In [5]:


s = Solver()
s.set('timeout', 900000)
s.add(*IGuarded.guards([l1, l2]), sublist(l1, l2))#, Not(sublist(l2, l1)))
#s.add(sublist(l2, l1))
    
s.add(card_geq(l1.dom, 1))
s.add(card_geq(l2.dom, 2))

def is_sorted(l: ListRef, R: Callable[ExprRef, ExprRef, BoolRef]):
    i, j = Elements('i h', l.dom)
    return ForAll([i, j], l.ord[i][j] >> R(l.val[i], l.val[j]))

s.add(is_sorted(l2, lambda x, y: x <= y))

# a ∈ l₂, ∀ e ∈ l₁, e ≤ a
ai = Element('ai', l2.dom)
s.add(*IGuarded.guards([ai]),
      Prod([l1.dom], lambda e: l1.val[e] <= l2.val[ai]))
      
# l₃ = l₁ ++ [a]
f1 = Mapping('f1', l1.dom ** l3.dom)
aj = Element('aj', l3.dom)
s.add(*IGuarded.guards([f1, aj, l3]),
      substitute_vars(sublist(l1, l3).body(), f1),
      l3.val[aj] == l2.val[ai],
      Prod([l3.dom], lambda e3: one_of(
          Sig([l1.dom], lambda e1: f1[e1] == e3), aj == e3)),
      Prod([l1.dom], lambda i: l3.ord[f1[i]][aj]))

# ¬l₃ <= l₂
if 1:
    f_tbl = Mapping('f_tbl', l3.dom ** l2.dom)
    f_tbl = tabulate(f_tbl)
    f_cexs = [
        tabulate(f_tbl, table={U[0]: U[1], U[2]: U[2], U[1]: U[0]}),
        tabulate(f_tbl, table={U[0]: U[1], U[1]: U[2], U[2]: U[0]}),
        tabulate(f_tbl, table={U[0]: U[2], U[1]: U[0], U[2]: U[1]}),
        tabulate(f_tbl, table={U[0]: U[2], U[1]: U[1], U[2]: U[0]}),
        tabulate(f_tbl, table={U[0]: U[0], U[1]: U[1], U[2]: U[2]}),
        tabulate(f_tbl, table={U[0]: U[0], U[1]: U[2], U[2]: U[1]})
    ]
    
    s.add(
        *(Not(substitute_vars(sublist(l3, l2).body(), f_tbl)) for f_tbl in f_cexs)
        #Not(Exists(list(f_tbl.table.values()), substitute_vars(sublist(l3, l2).body(), f_tbl)))
    )


#s.add(disjoint(l1.dom, l2.dom))

m = s.model() if (r := s.check()) == sat else r
m

unknown

In [8]:
#m.eval(l1.dom.set), m.eval(l2.dom.set), m.eval(f_id)

((ordered(l1.dom, l1.ord, m), l1 @ m),
 (ordered(l2.dom, l2.ord, m), l2 @ m),
 (ordered(l3.dom, l3.ord, m), l3 @ m), (ai @ m, m.eval(l2.val[ai])), (aj @ m, m.eval(l3.val[aj])), f1 @ m)

(([②], [8947]),
 ([①, ⓪], [8946, 8947]),
 ([②, ①], [8947, 8947]),
 (⓪, 8947),
 (①, 8947),
 {②: ②})

In [13]:
s.assertions()[0].body().ctx.guarded

In [ ]:
[z3.z3util.get_vars( a ) for a in s.assertions()]

In [ ]:
diag = [
    d() == m.get_interp(d) for d in m.decls() if d.arity() == 0]

diag
s_aux = Solver()
s_aux.add(*diag)
s_aux.add(sublist(l2, l1))
s_aux.check()

In [ ]:
with open('outf.smt2', 'w') as outf: outf.write(s.to_smt2())

In [ ]:
U.values()

In [ ]:
U[0]